# Identification du quadrirotor par réseau de neurones (FNN)

**Projet EI Data-Driven Control, Track B, étape 3.** On identifie la dynamique verticale du drone à partir des seules données mesurées (`bdd_in_mat_05.csv`, `bdd_out_mat_05.csv`), pour l'utiliser ensuite comme simulateur dans l'environnement d'apprentissage par renforcement.

Le notebook se lit comme un rapport : chaque cellule de code est précédée de ce qu'elle fait et de **pourquoi**. Les fonctions du modèle sont dans `fnn.py` (importé ici), pour que `env.py` puisse charger le modèle sans ce notebook.

**Plan :** 1. données et nettoyage → 2. choix des entrées du réseau (raisonnement physique) → 3. entraînement (comparaison des entrées, courbe d'apprentissage) → 4. validation en simulation libre sur le jeu de test → 5. conclusion.

**Options** (cellule suivante) :
- `RETRAIN = True` entraîne le modèle retenu (6 époques) et l'enregistre dans `model/fnn.pt` ; `False` recharge le modèle enregistré.
- `RUN_COMPARISON = True` refait la comparaison des 4 jeux d'entrées (3 époques chacun) ; `False` relit le résultat enregistré dans `model/input_comparison.csv`.

Durée totale avec les deux options à `True` : environ 10 minutes sur CPU.

In [ ]:
RETRAIN = True
RUN_COMPARISON = True

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from IPython.display import Markdown, display

import fnn   # model functions: build_fnn, make_regressors, train_fnn, predict, rollout, save_fnn, load_fnn

cfg = yaml.safe_load(open("fnn_config.yaml"))
SEED, TS = cfg["seed"], cfg["Ts"]
np.random.seed(SEED)
torch.manual_seed(SEED)
Path("figures").mkdir(exist_ok=True)
Path("model").mkdir(exist_ok=True)

# one color per entity, the same in every figure
C_MEAS, C_FNN, C_U, C_MUTED = "#3d3c39", "#2a78d6", "#1baf7a", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "figure.dpi": 110,
                     "axes.titlesize": 11, "legend.fontsize": 9})
print("seed", SEED, "| Ts", TS, "s | torch", torch.__version__)

## 1. Données et nettoyage (TEMPORARY)

**Ce qu'on sait des fichiers (vérifié)** : deux CSV sans en-tête de 3000 lignes × 2000 colonnes. Chaque **ligne est une trajectoire** indépendante, chaque colonne un instant. L'entrée vaut dans $[-1, 1]$, constante par paliers ; chaque trajectoire commence par 40 échantillons à $u = 0$.

**Hypothèses de travail (non vérifiées)** :
- **H1** : l'entrée est la commande de poussée normalisée $u$, la sortie l'accélération verticale $a_z$ en m/s², axe $z$ vers le haut. Indice fort : pendant les 40 échantillons à $u = 0$, la sortie vaut $-9.81$ m/s², c'est-à-dire $-g$ (chute libre).
- **H2** : les CSV sont échantillonnés à $T_s = 0.05$ s (suffixe `_05`), donc une trajectoire dure 100 s.

In [ ]:
U_raw = pd.read_csv(cfg["data"]["input_csv"], header=None).to_numpy(float)
AZ_raw = pd.read_csv(cfg["data"]["output_csv"], header=None).to_numpy(float)
print("input  shape", U_raw.shape, "| u range [%.3f, %.3f]" % (U_raw.min(), U_raw.max()))
print("output shape", AZ_raw.shape, "| az range [%.1f, %.1f]" % (AZ_raw.min(), AZ_raw.max()))
rest = AZ_raw[:, :40]                      # u = 0 on the first 40 samples of every trajectory
print("az at rest (u = 0): median %.3f m/s^2, typical std %.3f m/s^2" % (np.median(rest), np.median(rest.std(1))))

**Nettoyage minimal.** Le capteur est annoncé peu fiable. On repère trois défauts grossiers :
- valeurs non finies ;
- $|a_z| > 40$ m/s², alors que les données saines restent dans $[-30, 11]$ ;
- au moins 5 valeurs **exactement** identiques d'affilée, ce qui signale un capteur bloqué : avec un bruit de 0.04 m/s², deux mesures saines ne sont jamais égales.

Une trajectoire touchée est retirée **en entier** plutôt que réparée : on a 3000 trajectoires, en perdre 4 ne coûte rien, et interpoler introduirait des données inventées.

Ce nettoyage est **TEMPORARY** : il sera remplacé par la validation complète de l'étape 2 (dérives, normalisation, etc.). Il suffira de remplacer `U`, `AZ`, `VALID`, `TRAJ_ID` par leurs données.

In [ ]:
c = cfg["data"]["tmp_cleaning"]

def frozen_mask(y, min_len):
    # True on samples belonging to a run of >= min_len exactly equal consecutive values
    mask = np.zeros(y.shape, bool)
    for r in range(y.shape[0]):
        same = np.r_[False, y[r, 1:] == y[r, :-1]]
        edges = np.diff(np.r_[0, same.astype(int), 0])
        for s, e in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
            if e - s + 1 >= min_len:
                mask[r, s - 1:e] = True
    return mask

nonfinite = ~np.isfinite(U_raw) | ~np.isfinite(AZ_raw)
outlier = np.abs(np.nan_to_num(AZ_raw)) > c["az_abs_max"]
frozen = frozen_mask(AZ_raw, c["frozen_min_len"])
bad_rows = np.flatnonzero((nonfinite | outlier | frozen).any(axis=1))
report = pd.DataFrame({"row": bad_rows,
                       "frozen samples": frozen[bad_rows].sum(1),
                       "outliers |az|>40": outlier[bad_rows].sum(1),
                       "first bad sample": [np.flatnonzero((outlier | frozen)[r])[0] for r in bad_rows]})
display(report)
assert bad_rows.tolist() == sorted(c["known_bad_rows"]), "unexpected defects: check the data"

keep = np.setdiff1d(np.arange(len(U_raw)), bad_rows)
U, AZ, TRAJ_ID = U_raw[keep], AZ_raw[keep], keep
VALID = ~(nonfinite | outlier | frozen)[keep]
print(f"kept {len(keep)} trajectories out of {len(U_raw)}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, r, title in zip(axs, [0, 122, 455], ["healthy trajectory", "frozen sensor", "random junk"]):
    ax.plot(np.arange(200) * TS, AZ_raw[r, :200], color=C_MEAS, lw=1.2)
    ax.set(title=f"row {r}: {title}", xlabel="t [s]", ylabel="az [m/s²]")
fig.suptitle("Examples of the defects removed (first 10 s)", y=1.03)
plt.tight_layout(); plt.show()

**Découpage train / validation / test : par trajectoire entière (70 / 15 / 15 %), tirage avec seed.**

On ne découpe jamais point par point. Deux échantillons voisins d'une même trajectoire sont presque identiques : si l'un était en apprentissage et l'autre en test, le score de test serait artificiellement bon (fuite de données).

- **apprentissage** : ajuste les poids du réseau ;
- **validation** : sert à **choisir** (les entrées du réseau en section 3.1, le nombre d'époques en section 3.2) ;
- **test** : n'est utilisé qu'une fois, en section 4, pour mesurer la qualité du modèle final sur des trajectoires qui n'ont servi à aucun choix.

In [ ]:
perm = np.random.default_rng(SEED).permutation(len(U))
fr = cfg["data"]["split"]
n_tr, n_va = round(fr["train"] * len(U)), round(fr["val"] * len(U))
TRAIN, VAL, TEST = np.sort(perm[:n_tr]), np.sort(perm[n_tr:n_tr + n_va]), np.sort(perm[n_tr + n_va:])
print(f"trajectories: train {len(TRAIN)}, validation {len(VAL)}, test {len(TEST)}")

## 2. Choix des entrées du réseau : raisonnement physique

Le réseau prédit l'accélération à l'instant suivant, $a_z[k+1]$. La question est : **de quelles grandeurs dépend-elle physiquement ?** On compare quatre jeux d'entrées.

**Rappel sur le temps discret.** À l'instant $k$, on mesure $a_z[k]$, puis la commande $u[k]$ est appliquée et **maintenue pendant toute la période** $[t_k,\, t_{k+1}[$. La mesure $a_z[k+1]$ est prise à la fin de cette période. La commande $u[k]$ est donc bien dans le passé de $a_z[k+1]$.

| Entrées | Nombre | Hypothèse physique |
|---|---|---|
| A : $a_z[k],\ u[k-1]$ | 2 | Le moteur réagit en **1er ordre**, avec un **temps mort de 2 pas** (0.1 s) : la commande $u[k]$ n'aurait aucun effet sur $a_z[k+1]$ et n'agirait qu'à partir de $a_z[k+2]$. Cela suppose une latence supplémentaire, par exemple dans la transmission de la commande. |
| B : $a_z[k],\ u[k]$ | 2 | Le moteur réagit en **1er ordre**, avec un **retard d'1 pas** : la commande n'a aucun effet instantané, son effet apparaît à l'échantillon suivant. C'est exactement ce que donne un moteur du 1er ordre échantillonné avec une commande maintenue sur chaque période. |
| C : $a_z[k],\ u[k],\ u[k-1]$ | 3 | Même dynamique que B, mais la commande précédente a encore une influence propre : retard compris entre 1 et 2 pas, ou actionneur plus lent qu'un 1er ordre. |
| D : $a_z[k],\ a_z[k-1],\ u[k],\ u[k-1]$ | 4 | Dynamique du **2e ordre** : l'accélération dépend aussi de sa propre variation récente, par exemple moteur + filtre du capteur, ou un mode oscillant. |

**Pourquoi $a_z[k]$ est toujours en entrée.** À $u$ fixé, l'accélération ne saute pas d'une valeur à l'autre : le rotor a une inertie, et sa poussée évolue progressivement. $a_z[k]$ résume l'état actuel de la poussée (à $g$ près) : c'est la « mémoire » du système.

**Pourquoi pas la vitesse ou l'altitude.** On suppose que la poussée ne dépend ni de $v_z$ ni de $z$ : traînée aérodynamique négligeable et pas d'effet de sol dans les données (hypothèse). Le sol et la plateforme sont gérés dans l'environnement RL, pas dans ce modèle.

**Nombre de paramètres.** Seule la première couche du réseau dépend du nombre d'entrées $n$ : le total vaut $64n + 4289$ (cellule suivante). Passer de 2 à 4 entrées n'ajoute que 128 paramètres, soit moins de 3 %. Le choix ne se joue donc pas sur la taille du réseau mais sur **la structure physique** : ordre de la dynamique et valeur du retard.

**Règle de décision.** Le choix ne peut se faire qu'après l'entraînement (section 3), sur le jeu de **validation**. Si plusieurs jeux d'entrées donnent des erreurs proches, on garde celui qui a **le moins d'entrées** (principe de parcimonie) : un modèle plus simple est plus facile à interpréter et à défendre, et son état est plus petit dans l'environnement RL.

In [ ]:
VARIANTS = {   # name: (az lags, u lags) -> inputs az[k - l] for l in az lags, then u[k - l] for l in u lags
    "A: az[k], u[k-1]":               ([0], [1]),
    "B: az[k], u[k]":                 ([0], [0]),
    "C: az[k], u[k], u[k-1]":         ([0], [0, 1]),
    "D: az[k], az[k-1], u[k], u[k-1]": ([0, 1], [0, 1]),
}
n_params = {name: sum(p.numel() for p in fnn.build_fnn(cfg["fnn"], len(a) + len(b)).parameters())
            for name, (a, b) in VARIANTS.items()}
display(pd.DataFrame({"inputs": [len(a) + len(b) for a, b in VARIANTS.values()],
                      "parameters": list(n_params.values())}, index=list(VARIANTS)))

## 3. Entraînement

**Choix communs à tous les entraînements** (repris de l'étude précédente) :
- **Réseau** : couches cachées 64 → 64, activation tanh. C'est largement assez pour une fonction lisse de 2 à 4 variables, et très peu face aux ~4 millions d'exemples, donc sans risque de surapprentissage. tanh donne une fonction lisse, ce qui aide quand le modèle est réappliqué 200 fois en simulation.
- **Normalisation** : entrées et sortie centrées-réduites avec les statistiques du seul jeu d'apprentissage. Toutes les valeurs de $a_z$ (en entrée et en sortie) partagent les mêmes statistiques, car c'est la même grandeur physique. Ces statistiques sont enregistrées avec les poids.
- **Coût et optimiseur** : MSE (régression), Adam avec lr = 1e-3 et des lots de 1024 exemples. Même seed pour tous les entraînements.

### 3.1 Comparaison des jeux d'entrées (3 époques, jeu de validation)

Chaque jeu d'entrées est entraîné **3 époques**, avec exactement les mêmes réglages. On mesure ensuite l'erreur **en simulation libre sur 10 s (rollout)** : le réseau ne reçoit que l'état initial réel, puis il est rebouclé sur ses propres prédictions pendant 200 pas, avec la vraie séquence de commandes. C'est exactement le fonctionnement dans l'environnement RL, donc c'est le critère qui compte.

⚠️ **Cette comparaison est faite sur le jeu de VALIDATION** (2000 fenêtres de 10 s tirées au hasard dans les trajectoires de validation). Le jeu de test n'est pas utilisé ici : il est réservé à la mesure finale de la section 4.

In [ ]:
def regressors(rows, az_lags, u_lags):
    # pairs X = [az[k - l]..., u[k - l]...], y = az[k + 1], all samples valid, never across trajectories
    m = max(az_lags + u_lags)
    ks = np.arange(m, AZ.shape[1] - 1)
    A, Uu, V = AZ[rows], U[rows], VALID[rows]
    X = np.stack([A[:, ks - l] for l in az_lags] + [Uu[:, ks - l] for l in u_lags], axis=-1)
    ok = V[:, ks + 1].copy()
    for l in range(m + 1):
        ok &= V[:, ks - l]
    return X[ok], A[:, ks + 1][ok]


def random_windows(rows, H, n, seed):
    # n random fully valid windows: measured az[k-1..k+H] and u[k-1..k+H-1]
    rng = np.random.default_rng(seed)
    az_w, u_w = [], []
    while len(az_w) < n:
        r, k = rows[rng.integers(len(rows))], rng.integers(1, AZ.shape[1] - H)
        if VALID[r, k - 1:k + H + 1].all():
            az_w.append(AZ[r, k - 1:k + H + 1]); u_w.append(U[r, k - 1:k + H])
    return np.array(az_w), np.array(u_w)


def rollout_rmse(model, az_lags, u_lags, az_w, u_w):
    # free run: the network only gets the measured az at the start, then its own predictions
    H = u_w.shape[1] - 1
    P = az_w.copy()
    P[:, 2:] = np.nan                   # P[:, j] = az at sample k - 1 + j; only az[k-1], az[k] are known
    for j in range(1, H + 1):           # predict az[k + j] from the "current" sample k - 1 + j
        X = np.stack([P[:, j - l] for l in az_lags] + [u_w[:, j - l] for l in u_lags], axis=-1)
        P[:, j + 1] = fnn.f_phys(model, X)
    return float(np.sqrt(np.mean((P[:, 2:] - az_w[:, 2:]) ** 2)))


H = cfg["eval"]["horizon"]
az_val_w, u_val_w = random_windows(VAL, H, cfg["eval"]["n_windows"], SEED + 1)
comp_path = Path("model/input_comparison.csv")
if RUN_COMPARISON or not comp_path.exists():
    rows = []
    for name, (al, ul) in VARIANTS.items():
        print(f"--- {name}")
        Xtr, ytr = regressors(TRAIN, al, ul)
        Xva, yva = regressors(VAL, al, ul)
        m, _ = fnn.train_fnn(cfg["fnn"], Xtr, ytr, Xva, yva, TS, cfg["fnn"]["compare_epochs"],
                             seed=SEED, n_az=len(al))
        rows.append({"inputs": name, "n inputs": len(al) + len(ul), "parameters": n_params[name],
                     "rollout RMSE 10 s, validation [m/s²]": rollout_rmse(m, al, ul, az_val_w, u_val_w)})
    comparison = pd.DataFrame(rows)
    comparison.to_csv(comp_path, index=False)
else:
    comparison = pd.read_csv(comp_path)
    print("loaded", comp_path)

col = "rollout RMSE 10 s, validation [m/s²]"
best = comparison[col].min()
TOL = 0.10    # "close" = within 10 % of the best error
close = comparison[comparison[col] <= (1 + TOL) * best]
chosen = close.sort_values(["n inputs", col]).iloc[0]["inputs"]
comparison["relative to best"] = comparison[col] / best
display(comparison.style.format({col: "{:.4f}", "relative to best": "{:.2f}×"}).hide(axis="index"))
print(f"retained (fewest inputs within {TOL:.0%} of the best): {chosen}")
if not chosen.startswith("B"):
    print("WARNING: with this rule the retained inputs are not B. The rest of the notebook still uses B "
          "(structure validated with the supervisor, the only one supported by fnn.predict / rollout): "
          "discuss this result before going further.")


**Lecture du tableau.** On retient le jeu d'entrées qui a le moins d'entrées parmi ceux dont l'erreur est à moins de 10 % de la meilleure. Le seuil de 10 % est un choix de notre part : avec une seule seed et 3 époques, un écart plus petit n'est pas significatif.

Ce qu'on attend physiquement, et ce que le tableau permet de vérifier :
- **A** doit être nettement moins bon : s'il n'y a pas de temps mort de 2 pas, le réseau n'a pas accès à la commande qui agit vraiment, $u[k]$.
- **B, C et D** doivent être proches : si le moteur est bien un 1er ordre avec un retard d'1 pas, $u[k-1]$ et $a_z[k-1]$ n'apportent pas d'information supplémentaire.

Si c'est le cas, on retient **B : $a_z[k+1] = f(a_z[k],\, u[k])$**. C'est l'écriture compacte de la structure validée avec l'encadrant : la commande et l'accélération du pas précédent déterminent l'accélération courante, $a_z[k] = f(a_z[k-1],\, u[k-1])$. Ce sont exactement les grandeurs `az_prev` et `u_prev` que reçoit le contrôleur dans `rl_controller.m`.

### 3.2 Courbe d'apprentissage du modèle retenu (époques 1 à 6)

On entraîne le modèle retenu sur 6 époques. Après chaque époque, on mesure la RMSE (en m/s²) sur **tout** le jeu d'apprentissage et sur **tout** le jeu de validation, en prédiction à un pas.

- Si l'erreur d'apprentissage baisse alors que celle de validation remonte, c'est du surapprentissage.
- Si les deux restent proches et se stabilisent, le réseau a convergé.

On garde les poids de l'époque où l'erreur de **validation** est la plus faible : c'est ainsi que l'on choisit le nombre d'époques optimal. La ligne horizontale indique le bruit du capteur, en dessous duquel aucun modèle ne peut descendre.

In [ ]:
noise_std = float(np.median(AZ[TRAIN, :40].std(axis=1)))     # u = 0 on the first 40 samples: pure sensor noise
X_tr, y_tr = fnn.make_regressors(U[TRAIN], AZ[TRAIN], VALID[TRAIN])
X_va, y_va = fnn.make_regressors(U[VAL], AZ[VAL], VALID[VAL])
print(f"training pairs {len(X_tr):,} | validation pairs {len(X_va):,}")

model_path = Path(cfg["fnn"]["model_path"])
if RETRAIN or not model_path.exists():
    model, history = fnn.train_fnn(cfg["fnn"], X_tr, y_tr, X_va, y_va, TS, cfg["fnn"]["n_epochs"], seed=SEED)
    model["meta"]["test_traj_id"] = TRAJ_ID[TEST].tolist()
    fnn.save_fnn(model, model_path)
    print("saved", model_path)
else:
    model = fnn.load_fnn(model_path)
    history = model["meta"]["history"]
    print("loaded", model_path)
assert model["meta"]["test_traj_id"] == TRAJ_ID[TEST].tolist(), "test split differs from the saved model"
best_epoch = model["meta"]["best_epoch"]

h = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(h["epoch"], h["train_rmse"], "o-", color=C_FNN, label="training")
ax.plot(h["epoch"], h["val_rmse"], "s--", color=C_MEAS, label="validation")
ax.axhline(noise_std, color=C_MUTED, ls=":", lw=1)
ax.text(h["epoch"].iloc[-1], noise_std, " sensor noise", va="bottom", ha="right", fontsize=9, color=C_MUTED)
ax.axvline(best_epoch, color=C_MUTED, ls=":", lw=1)
ax.set(xlabel="epoch", ylabel="one-step RMSE [m/s²]", xticks=h["epoch"],
       title=f"Learning curve, inputs az[k], u[k] (seed {SEED}), best validation at epoch {best_epoch}")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("figures/learning_curve.png", dpi=150); plt.show()
display(h.set_index("epoch").style.format("{:.4f}"))

**Lecture de la courbe.** Avec plus de 4 millions d'exemples, une époque représente déjà environ 4 000 mises à jour des poids : l'erreur baisse donc très vite dès les premières époques. Tant que les erreurs d'apprentissage et de validation restent superposées, il n'y a pas de surapprentissage. Si l'erreur remonte ou oscille aux dernières époques, cela vient du pas d'apprentissage constant (lr = 1e-3), qui fait encore bouger les poids, et non d'un surapprentissage. On garde les poids de l'époque où l'erreur de validation est minimale (indiquée dans le titre) : c'est le nombre d'époques optimal.

## 4. Validation en simulation libre sur le jeu de test

Le modèle retenu est maintenant évalué **une seule fois** sur les trajectoires de **test**, qui n'ont servi à aucun choix (ni entrées, ni nombre d'époques).

**Test en simulation libre sur 10 s (rollout).** Le modèle ne reçoit que l'état initial mesuré $[a_z[k],\, u[k-1]]$, puis il est rebouclé sur ses propres prédictions pendant 200 pas, avec la vraie séquence de commandes. Les erreurs peuvent s'accumuler, comme dans l'environnement RL.

La figure montre une trajectoire de test : l'accélération mesurée et celle du FNN, l'erreur entre les deux, et la commande. La RMSE est ensuite calculée sur 2000 fenêtres de 10 s tirées au hasard dans les trajectoires de test.

In [ ]:
r = next(r for r in TEST if VALID[r].all())          # plotted test trajectory
k0 = int(cfg["eval"]["plot_start_s"] / TS)
ks = np.arange(k0, k0 + H)
meas = AZ[r, ks + 1]
pred = fnn.rollout(model, fnn.state_at(AZ[r], U[r], k0)[None], U[r, ks][None])[0]
t = (ks + 1) * TS

fig, (a0, a1, a2) = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True, gridspec_kw={"height_ratios": [3, 1.3, 1]})
a0.plot(t, meas, color=C_MEAS, lw=2.4, label="measured (system)")
a0.plot(t, pred, color=C_FNN, lw=1.3, label="FNN, free run")
a0.set(ylabel="az [m/s²]", title=f"Free-run simulation over {H * TS:.0f} s, test trajectory #{TRAJ_ID[r]} "
                                 f"(seed {SEED}), RMSE {np.sqrt(np.mean((pred - meas) ** 2)):.3f} m/s²")
a0.legend(frameon=False, loc="best")
a1.plot(t, pred - meas, color=C_FNN, lw=1)
a1.axhspan(-noise_std, noise_std, color=C_MUTED, alpha=0.15, lw=0, label="± sensor noise")
a1.axhline(0, color=C_MUTED, lw=0.8)
a1.set(ylabel="FNN − measured\n[m/s²]"); a1.legend(frameon=False, loc="lower right")
a2.step(t - TS, U[r, ks], where="post", color=C_U)
a2.set(ylabel="command u [-]", xlabel="time t [s]", ylim=(-1.1, 1.1))
plt.tight_layout(); plt.savefig("figures/overlay_rollout.png", dpi=150); plt.show()

az_te_w, u_te_w = random_windows(TEST, H, cfg["eval"]["n_windows"], SEED + 2)
test_rollout_rmse = rollout_rmse(model, [0], [0], az_te_w, u_te_w)
print(f"rollout RMSE over {len(az_te_w)} random 10 s test windows: {test_rollout_rmse:.4f} m/s^2 "
      f"(sensor noise {noise_std:.4f} m/s^2)")

## 5. Conclusion

In [ ]:
rB = comparison.set_index("inputs")[col]
display(Markdown(rf'''
**Modèle retenu : $a_z[k+1] = f\big(a_z[k],\, u[k]\big)$**, soit $a_z[k] = f\big(a_z[k-1],\, u[k-1]\big)$ : un réseau feedforward à 2 entrées (64 → 64, tanh, {n_params['B: az[k], u[k]']} paramètres), entraîné {best_epoch} époque(s).

| | RMSE [m/s²] |
|---|---|
| bruit du capteur (référence) | {noise_std:.4f} |
| rollout 10 s, validation, entrées A : $a_z[k], u[k-1]$ | {rB.iloc[0]:.4f} |
| rollout 10 s, validation, entrées B : $a_z[k], u[k]$ (retenu) | {rB.iloc[1]:.4f} |
| rollout 10 s, validation, entrées C : $a_z[k], u[k], u[k-1]$ | {rB.iloc[2]:.4f} |
| rollout 10 s, validation, entrées D : $a_z[k], a_z[k-1], u[k], u[k-1]$ | {rB.iloc[3]:.4f} |
| **rollout 10 s, test, modèle final** | **{test_rollout_rmse:.4f}** |
'''))
gain_hist = 1 - min(rB.iloc[2], rB.iloc[3]) / rB.iloc[1]     # best improvement brought by more history
if chosen.startswith("B"):
    msg = (f"- **Structure physique** : un retard d'1 pas (la commande n'a pas d'effet instantané) et une dynamique "
           f"du 1er ordre. Ajouter de l'historique ($u[k-1]$, $a_z[k-1]$) change l'erreur de moins de {TOL:.0%} "
           f"(meilleur gain : {gain_hist:.0%}) : par parcimonie, on garde 2 entrées. Supposer un temps mort de 2 pas "
           f"(entrées A) multiplie l'erreur par {rB.iloc[0] / rB.iloc[1]:.1f}.")
else:
    msg = (f"- **À discuter** : avec la règle des {TOL:.0%}, les données préfèrent **{chosen}** (gain de {gain_hist:.0%} "
           f"sur B). Le modèle B est conservé ici car c'est la structure validée avec l'encadrant ; ce résultat "
           f"suggère que la dynamique n'est pas exactement du 1er ordre. Supposer un temps mort de 2 pas (entrées A) "
           f"multiplie l'erreur par {rB.iloc[0] / rB.iloc[1]:.1f}.")
display(Markdown(msg))

**Ce qu'il faut retenir (suite).**
- **Qualité** : en simulation libre sur 10 s, la durée de la mission, l'erreur reste du même ordre que le bruit du capteur et ne diverge pas. Le modèle est utilisable comme simulateur pour l'environnement RL.
- **Lien avec MATLAB** : l'état du modèle $[a_z[k],\, u[k-1]]$ correspond aux mesures `az_prev` et `u_prev` reçues par `rl_controller.m` (vérifié dans `EINT_RL_closed_loop.slx` : chacune passe par un bloc `UnitDelay`).

**Hypothèses à confirmer** : H1 ($u$ = commande de poussée, $a_z$ en m/s²), H2 ($T_s$ = 0.05 s), H3 (pas de contact avec le sol dans les données), H4 ($u[k]$ maintenue sur $[t_k, t_{k+1}[$).

**Limites** : une seule seed d'entraînement ; dans les données, $u$ reste constant au moins 5 pas, donc une politique qui change de commande à chaque pas sortirait du domaine appris (d'où une pénalité sur $\Delta u$ dans la reward) ; un petit biais sur $a_z$ devient une dérive sur $z$ après double intégration, que la politique corrige en boucle fermée.

**Utilisation dans `env.py`** (sans ce notebook) :
```python
from identification_fnn.fnn import load_fnn, predict
model = load_fnn()                         # identification_fnn/model/fnn.pt
state = [az_k, u_km1]                      # [az[k], u[k-1]], physical units
next_state = predict(model, state, u_k)    # [az[k+1], u[k]]
```